In [1]:
import pandas as pd
import torch

In [2]:
train_df = pd.read_csv("archive/train.csv")
test_df = pd.read_csv("archive/test.csv")
validation_df = pd.read_csv("archive/validation.csv")

In [3]:
train_df.head()
train_df.info()
train_df.isna().sum()
train_df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22593 entries, 0 to 22592
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Unnamed: 0          22593 non-null  int64 
 1   NCBIGeneID          22593 non-null  int64 
 2   Symbol              22593 non-null  object
 3   Description         22593 non-null  object
 4   GeneType            22593 non-null  object
 5   GeneGroupMethod     22593 non-null  object
 6   NucleotideSequence  22593 non-null  object
dtypes: int64(2), object(5)
memory usage: 1.2+ MB


Unnamed: 0            0
NCBIGeneID            0
Symbol                0
Description           0
GeneType              0
GeneGroupMethod       0
NucleotideSequence    0
dtype: int64

In [4]:
train_df.head()

,Unnamed: 0,NCBIGeneID,Symbol,Description,GeneType,GeneGroupMethod,NucleotideSequence
0,0,106481178,RNU4-21P,"RNA, U4 small nuclear 21, pseudogene",PSEUDO,NCBI Ortholog,<AGCTTAGCACAGTGGCAGTATCATAGGCAGTGAGGTTTATCCGAG...
1,1,123477792,LOC123477792,Sharpr-MPRA regulatory region 12926,BIOLOGICAL_REGION,NCBI Ortholog,<CTGGAGCGGCCACGATGTGAACTGTCACCGGCCACTGCTGCTCCG...
2,2,113174975,LOC113174975,Sharpr-MPRA regulatory region 7591,BIOLOGICAL_REGION,NCBI Ortholog,<TTCCCAATTTTTCCTCTGCTTTTTAATTTTCTAGTTTCCTTTTTC...
3,3,116216107,LOC116216107,CRISPRi-validated cis-regulatory element chr10...,BIOLOGICAL_REGION,NCBI Ortholog,<CGCCCAGGCTGGAGTGCAGTGGCGCCATCTCGGCTCACTGCAGGC...
4,4,28502,IGHD2-21,immunoglobulin heavy diversity 2-21,OTHER,NCBI Ortholog,<AGCATATTGTGGTGGTGACTGCTATTCC>


In [5]:
train_df["Description"].value_counts()

Description
small nucleolar RNA U13                  428
5.8S ribosomal RNA                       171
small nucleolar RNA U3                   106
U2 spliceosomal RNA                       99
5S ribosomal RNA                          86
                                        ... 
microRNA 548m                              1
Sharpr-MPRA regulatory region 12117        1
Nanog homeobox pseudogene 6                1
RNA, U6 small nuclear 485, pseudogene      1
Sharpr-MPRA regulatory region 11449        1
Name: count, Length: 21034, dtype: int64

In [6]:
train_df.columns

Index(['Unnamed: 0', 'NCBIGeneID', 'Symbol', 'Description', 'GeneType',
       'GeneGroupMethod', 'NucleotideSequence'],
      dtype='object')

In [7]:
train_df = train_df[["Description","NucleotideSequence"]]

In [8]:
train_df["prefix"] = train_df["Description"].str.extract(r'^([A-Za-z]+)')

In [9]:
train_df["Description"] = train_df["Description"].str.replace(r"[,]", "", regex=True)
train_df["NucleotideSequence"] = train_df["NucleotideSequence"].str.replace(r"[<>]", "", regex=True)

In [10]:
def divide_in_codons(df, sequence_column="NucleotideSequence", new_column="Codons"):
    train_df["Codons"] = train_df[sequence_column].apply(
        lambda tri: [tri[i:i+3] for i in range(0, len(tri), 3)]
    )
    return df

In [11]:
train_df.head()

,Description,NucleotideSequence,prefix
0,RNA U4 small nuclear 21 pseudogene,AGCTTAGCACAGTGGCAGTATCATAGGCAGTGAGGTTTATCCGAGG...,RNA
1,Sharpr-MPRA regulatory region 12926,CTGGAGCGGCCACGATGTGAACTGTCACCGGCCACTGCTGCTCCGA...,Sharpr
2,Sharpr-MPRA regulatory region 7591,TTCCCAATTTTTCCTCTGCTTTTTAATTTTCTAGTTTCCTTTTTCC...,Sharpr
3,CRISPRi-validated cis-regulatory element chr10...,CGCCCAGGCTGGAGTGCAGTGGCGCCATCTCGGCTCACTGCAGGCT...,CRISPRi
4,immunoglobulin heavy diversity 2-21,AGCATATTGTGGTGGTGACTGCTATTCC,immunoglobulin


In [12]:
train_df = divide_in_codons(train_df)

In [ ]:
def map_nucleotides(sequence):
    remaping_DNA = {"A":4,"G":5,"C":6,"T":7}
    return [remaping_DNA[letter] for letter in sequence]

NameError: name 'letter' is not defined

In [31]:
train_df["NucleotideSequence"]

0        AGCTTAGCACAGTGGCAGTATCATAGGCAGTGAGGTTTATCCGAGG...
1        CTGGAGCGGCCACGATGTGAACTGTCACCGGCCACTGCTGCTCCGA...
2        TTCCCAATTTTTCCTCTGCTTTTTAATTTTCTAGTTTCCTTTTTCC...
3        CGCCCAGGCTGGAGTGCAGTGGCGCCATCTCGGCTCACTGCAGGCT...
4                             AGCATATTGTGGTGGTGACTGCTATTCC
                               ...                        
22588    GGTGGGGTGGGGTGGGGTGGGGTGGGGTGCAGAGAAAACGATTGAT...
22589    GTGCTCACTTCAGCAGCACATATACTAAAATTGGAATGATACAGAG...
22590    GCTGGGCGTGGTGGTGGGTGCCTGTAATCCCAGCTACTAGGGAGGC...
22591    TCGTCCTGAAGCAGCGGCCAGAGAAGAGACAAGGGCACGAGCATCA...
22592    GGGGATGTAGCTCAGTGGTAGAGCGCATGCTTTGCATGTATGAGGC...
Name: NucleotideSequence, Length: 22593, dtype: object

In [14]:
train_df.fillna(0)

,Description,NucleotideSequence,prefix,Codons
0,RNA U4 small nuclear 21 pseudogene,AGCTTAGCACAGTGGCAGTATCATAGGCAGTGAGGTTTATCCGAGG...,RNA,"[AGC, TTA, GCA, CAG, TGG, CAG, TAT, CAT, AGG, ..."
1,Sharpr-MPRA regulatory region 12926,CTGGAGCGGCCACGATGTGAACTGTCACCGGCCACTGCTGCTCCGA...,Sharpr,"[CTG, GAG, CGG, CCA, CGA, TGT, GAA, CTG, TCA, ..."
2,Sharpr-MPRA regulatory region 7591,TTCCCAATTTTTCCTCTGCTTTTTAATTTTCTAGTTTCCTTTTTCC...,Sharpr,"[TTC, CCA, ATT, TTT, CCT, CTG, CTT, TTT, AAT, ..."
3,CRISPRi-validated cis-regulatory element chr10...,CGCCCAGGCTGGAGTGCAGTGGCGCCATCTCGGCTCACTGCAGGCT...,CRISPRi,"[CGC, CCA, GGC, TGG, AGT, GCA, GTG, GCG, CCA, ..."
4,immunoglobulin heavy diversity 2-21,AGCATATTGTGGTGGTGACTGCTATTCC,immunoglobulin,"[AGC, ATA, TTG, TGG, TGG, TGA, CTG, CTA, TTC, C]"
...,...,...,...,...
22588,uncharacterized LOC124907055,GGTGGGGTGGGGTGGGGTGGGGTGGGGTGCAGAGAAAACGATTGAT...,uncharacterized,"[GGT, GGG, GTG, GGG, TGG, GGT, GGG, GTG, GGG, ..."
22589,RNA U6 small nuclear 1060 pseudogene,GTGCTCACTTCAGCAGCACATATACTAAAATTGGAATGATACAGAG...,RNA,"[GTG, CTC, ACT, TCA, GCA, GCA, CAT, ATA, CTA, ..."
22590,RNA 7SL cytoplasmic 387 pseudogene,GCTGGGCGTGGTGGTGGGTGCCTGTAATCCCAGCTACTAGGGAGGC...,RNA,"[GCT, GGG, CGT, GGT, GGT, GGG, TGC, CTG, TAA, ..."
22591,NADH:ubiquinone oxidoreductase subunit S5 pseu...,TCGTCCTGAAGCAGCGGCCAGAGAAGAGACAAGGGCACGAGCATCA...,NADH,"[TCG, TCC, TGA, AGC, AGC, GGC, CAG, AGA, AGA, ..."


In [15]:
train_df["words_size"] = train_df["Description"].str.split().apply(lambda words: [len(w) for w in words])

In [16]:
word_lenght = train_df["words_size"].apply(pd.Series).rename(columns= lambda x: f"w_lenght_{x}")
train_df = pd.concat([train_df, word_lenght], axis=1).drop(columns=["words_size"])

In [25]:
def encoder_codons():
    for axes in train_df["Codons"]:
        n_codon = len(axes)
        print(n_codon)

In [26]:
encoder_codons()

42
99
99
92
10
19
32
176
28
35
88
161
35
96
89
299
99
36
31
99
35
99
52
25
39
115
263
99
310
84
20
99
234
99
213
97
304
92
276
35
203
131
36
294
41
99
99
65
24
99
99
99
20
281
99
99
320
19
134
22
99
167
193
24
99
29
222
189
99
99
22
163
333
61
203
322
174
221
82
37
114
39
36
120
99
118
302
242
63
98
123
99
99
37
99
55
250
140
99
99
101
323
99
99
36
212
32
112
21
133
99
99
35
99
24
224
33
105
99
99
132
213
99
99
99
36
36
167
35
99
275
34
99
171
99
30
186
99
97
99
25
224
215
23
52
99
99
27
205
70
41
99
23
243
30
191
28
100
101
292
24
275
30
36
34
99
192
209
99
281
30
99
126
35
99
243
233
107
99
207
99
185
25
99
92
33
99
70
155
49
168
126
125
116
67
30
36
99
28
99
44
25
99
325
74
36
31
103
57
32
24
19
177
36
49
37
21
27
21
249
239
234
25
273
95
99
26
98
210
48
49
221
36
58
206
241
323
127
275
165
8
46
175
100
99
50
273
167
310
28
99
191
99
210
99
72
96
31
217
68
121
40
26
36
314
37
245
31
20
99
25
99
99
99
309
21
80
172
312
321
99
99
35
306
266
118
63
36
24
36
17
244
35
276
36
99
38
26
57

In [17]:
# class PreparationData():
    # divide_in_codons(train_df)


In [ ]:
select_columns = ['prefix',
       'w_lenght_0', 'w_lenght_1', 'w_lenght_2', 'w_lenght_3', 'w_lenght_4',
       'w_lenght_5', 'w_lenght_6', 'w_lenght_7', 'w_lenght_8', 'w_lenght_9',
       'w_lenght_10', 'w_lenght_11', 'w_lenght_12', 'w_lenght_13',
       'w_lenght_14', 'NucleotideSequence']

In [22]:
X_train =  train_df[select_columns]
y_train  = train_df["Description"]
